# Preparing the Cleaned NLPCSS-20 Corpus

This notebook prepares the cleaned NLPCSS-20 corpus used for model application. The cleaning procedure removes:

1. the 45 manually annotated AllSides articles used for out-of-domain evaluation;
2. one extreme long noisy outlier;
3. articles shorter than 50 words;
4. exact duplicate article texts.

The resulting cleaned corpus is saved to `data/inputs/nlpcss20_clean.jsonl` and `data/inputs/nlpcss20_clean.csv`. A cleaning report is saved to `results/reports/nlpcss20_cleaning_report.csv`.

In [2]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "inputs"
REPORT_DIR = PROJECT_ROOT / "results" / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = RAW_DIR / "released_data.json"
VALIDATION_45_PATH = RAW_DIR / "validation_sample_45_articles.csv"

OUTPUT_JSONL = OUTPUT_DIR / "nlpcss20_clean.jsonl"
OUTPUT_CSV = OUTPUT_DIR / "nlpcss20_clean.csv"
CLEANING_REPORT = REPORT_DIR / "nlpcss20_cleaning_report.csv"

CORPUS_PATH, VALIDATION_45_PATH, OUTPUT_JSONL, OUTPUT_CSV, CLEANING_REPORT

(PosixPath('/home6/s4429621/thesis-modernBERT/data/raw/released_data.json'),
 PosixPath('/home6/s4429621/thesis-modernBERT/data/raw/validation_sample_45_articles.csv'),
 PosixPath('/home6/s4429621/thesis-modernBERT/data/inputs/nlpcss20_clean.jsonl'),
 PosixPath('/home6/s4429621/thesis-modernBERT/data/inputs/nlpcss20_clean.csv'),
 PosixPath('/home6/s4429621/thesis-modernBERT/results/reports/nlpcss20_cleaning_report.csv'))

## Load the Raw NLPCSS-20 Corpus

The raw corpus is loaded from `data/raw/released_data.json`. The original row position is stored as `original_index` so that the 45 manually annotated articles can be removed reliably. A simple word-count variable is also added for filtering short articles and identifying length outliers.

In [3]:
df = pd.read_json(CORPUS_PATH, lines=True)

df["original_index"] = df.index
df["word_count"] = df["content"].fillna("").str.split().str.len()

df.shape

(7775, 10)

## Remove the 45 Manually Annotated Articles

The 45 manually annotated AllSides articles were used for out-of-domain evaluation. They are therefore removed from the unannotated NLPCSS-20 corpus before model application. Removal is based on the `original_index` column.

In [4]:
validation_45 = pd.read_csv(VALIDATION_45_PATH)

validation_indices = set(validation_45["original_index"])

df_no_validation = df[~df["original_index"].isin(validation_indices)].copy()

df_no_validation.shape

(7730, 10)

## Remove the Extreme Long Outlier

The article with `original_index == 5311` is removed because it is an extreme length outlier and contains noisy content.

In [5]:
df_no_outlier = df_no_validation[df_no_validation["original_index"] != 5311].copy()

df_no_outlier.shape

(7729, 10)

## Remove Very Short Articles

Articles shorter than 50 words are removed because these texts are unlikely to contain enough substantive argumentative content for the corpus-level analysis.

In [6]:
df_no_short = df_no_outlier[df_no_outlier["word_count"] >= 50].copy()

df_no_short.shape

(7701, 10)

## Remove Exact Duplicate Article Texts

Exact duplicate article texts are removed based on the `content` column. The first occurrence is kept.

In [7]:
df_clean = df_no_short.drop_duplicates(subset="content", keep="first").copy()

df_clean.shape

(7656, 10)

## Save the Cleaned Corpus and Cleaning Report

The cleaned corpus is saved in both JSONL and CSV format. A separate cleaning report records the number of articles after each cleaning step.

In [8]:
df_clean.to_json(OUTPUT_JSONL, orient="records", lines=True, force_ascii=False)
df_clean.to_csv(OUTPUT_CSV, index=False)

cleaning_report = pd.DataFrame([
    {"step": "original_corpus", "articles": len(df)},
    {"step": "after_validation_article_removal", "articles": len(df_no_validation)},
    {"step": "after_extreme_outlier_removal", "articles": len(df_no_outlier)},
    {"step": "after_short_article_removal", "articles": len(df_no_short)},
    {"step": "after_exact_duplicate_removal", "articles": len(df_clean)},
])

cleaning_report.to_csv(CLEANING_REPORT, index=False)

cleaning_report

,step,articles
0,original_corpus,7775
1,after_validation_article_removal,7730
2,after_extreme_outlier_removal,7729
3,after_short_article_removal,7701
4,after_exact_duplicate_removal,7656


## Verification

The final checks verify that the cleaned corpus excludes the 45 manually annotated articles, excludes the extreme outlier, contains no articles below 50 words, and contains no exact duplicate article texts.

In [9]:
clean_indices = set(df_clean["original_index"])

print("Raw NLPCSS-20 rows:", len(df))
print("Clean NLPCSS-20 rows:", len(df_clean))
print()

print("Validation 45 articles still in clean corpus:", len(validation_indices & clean_indices))
print("Validation overlaps:", sorted(validation_indices & clean_indices))
print()

print("Extreme outlier original_index 5311 still in clean corpus:", 5311 in clean_indices)
print()

print("Shortest article in clean corpus, word count:", df_clean["word_count"].min())
print("Articles below 50 words in clean corpus:", int((df_clean["word_count"] < 50).sum()))
print()

print("Exact duplicate article texts in clean corpus:", int(df_clean.duplicated(subset="content").sum()))

Raw NLPCSS-20 rows: 7775
Clean NLPCSS-20 rows: 7656

Validation 45 articles still in clean corpus: 0
Validation overlaps: []

Extreme outlier original_index 5311 still in clean corpus: False

Shortest article in clean corpus, word count: 50
Articles below 50 words in clean corpus: 0

Exact duplicate article texts in clean corpus: 0
